In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import sys
import time
import datetime
import seaborn as sns
import matplotlib.pyplot as plt
from termcolor import colored, cprint


import sys
from os import listdir
from os.path import isfile, join

import numpy as np
import pandas as pd  # Import Pandas and Numpy to create databases
import tensorflow as tf  # Import tensorflow to do tensor manipulations

from abyss.helpers import metrics_viewer
from abyss.models import create_autoencoder
from abyss.glob import glob

from sklearn.metrics import silhouette_score
from sklearn.preprocessing import LabelEncoder

In [ ]:
path_data = f"data/phenotype"
humans = pd.read_pickle(f"{path_data}/all.panel")

In [ ]:
# Parse command line arguments
args = sys.argv
# Set default values if arguments are not provided
epoch = 15
bottleneck = 2
PCs = 2
bin = 1

In [ ]:
# Define paths and directories
PATH_GENO = glob.PATH_GENO
path_input = f"{PATH_GENO}/07_lowest_dimensionality"

In [ ]:
list_of_bins = os.listdir(path_input)

In [ ]:
bins = []
dims_to_append = []
silhouettes = []

for bin in list_of_bins:
    path_files = path_input + f"/{bin}/{bottleneck}_dim_latent"
    reduction_files = [f for f in os.listdir(path_files) if not f.startswith("metrics")]
    for file in reduction_files:
        nr_dims = int(file.split("_")[3])
        path_reduction = f"{path_files}/{file}"
        dims = pd.read_pickle(path_reduction)
        label_encoder = LabelEncoder()
        temp = dims.copy()
        temp['pop'] = list(humans['Superpopulation code'])

        # Using the encoded labels in silhouette_score
        silhouette_pop = silhouette_score(dims, temp['pop'])
        print("Silhouette Score:", silhouette_pop)
        #if nr_dims == 2:
            # Scatter plot with coloring by 'population'
            #sns.scatterplot(data=temp, x='dim1', y='dim2', hue='pop', palette='rocket')
            #plt.xlabel('dim1')
            #plt.ylabel('dim2')
            #plt.title(f"Silhouette score = {silhouette_pop}")
            #plt.show()
        #else:
            #pass
        bins.append(bin)
        dims_to_append.append(nr_dims)
        silhouettes.append(silhouette_pop)
        

In [ ]:
# Creating a DataFrame
df = pd.DataFrame({
    'Bin': bins,
    'Dimension': dims_to_append,
    'Silhouette Score': silhouettes
})


In [ ]:

# Extracting the numeric part of the 'Bin' column
df['Bin'] = df['Bin'].str.extract('(\d+)').astype(int)

# Creating the boxplot
sns.boxplot(x='Bin', y='Silhouette Score', data=df)

# Setting the title and labels for clarity
plt.title('Boxplot of Silhouette Scores for Different Bins')
plt.xlabel('Bin Number')
plt.ylabel('Silhouette Score')

# Display the plot
plt.show()

In [ ]:
# Creating the scatter plot
sns.scatterplot(x='Bin', y='Silhouette Score', hue='Dimension', data=df, palette="rocket")

# Setting the title and labels for clarity
plt.title('Scatter Plot of Silhouette Scores by Bin and Dimension')
plt.xlabel('Bin Number')
plt.ylabel('Silhouette Score')

# Display the plot
plt.show()

In [ ]:
# Creating the line plot
sns.lineplot(x='Bin', y='Silhouette Score', hue='Dimension', data=df, marker="o")

# Setting the title and labels for clarity
plt.title('Line Plot of Silhouette Scores by Bin and Dimension')
plt.xlabel('Bin Number')
plt.ylabel('Silhouette Score')

# Display the plot
plt.show()

In [ ]:
# Creating the line plot
line_plot = sns.lineplot(x='Bin', y='Silhouette Score', hue='Dimension', data=df, marker="o")

# Adjusting the legend
plt.legend(title='Dimension', loc='best', ncol=1)

# Setting the title and labels for clarity
plt.title('Line Plot of Silhouette Scores by Bin and Dimension')
plt.xlabel('Bin Number')
plt.ylabel('Silhouette Score')

# Display the plot
plt.show()